# Train a YOLO Number Plate Detector (Indian plates) — Google Colab

**Course:** Deep Learning &nbsp;|&nbsp; **Tools:** Python, PyTorch, Ultralytics YOLO, OpenCV, KaggleHub
**Dataset:** `kedarsai/indian-license-plates-with-labels` (YOLO-format boxes, one class)

## What you will do

1. Download and **audit** the dataset (missing labels, bad boxes, duplicates, box-size distribution).
2. Build a clean train / val / test split.
3. Fine-tune a pretrained YOLO detector (transfer learning, same idea as the pretrained Faster R-CNN in Lab 11).
4. Evaluate with precision, recall, mAP and choose the best **confidence threshold**.
5. **Stress-test** on road-like conditions (small, blurred, dark plates) because the dataset is mostly close-ups.
6. Export `best.pt` for the VS Code real-time project.

## Pipeline

`Kaggle dataset → audit → split → YOLO fine-tuning → evaluation → stress test → best.pt → real-time app`

> **Runtime → Change runtime type → T4 GPU** before you start.
> A detector only **locates** plates. Reading the characters is done later by an OCR model in the VS Code project.

## 1. Check the GPU and install packages

In [ ]:
# What this cell does: Shows the GPU and installs Ultralytics (YOLO) and KaggleHub.
# How it works: nvidia-smi prints the GPU; pip installs the two packages (PyTorch/OpenCV are already on Colab).
!nvidia-smi -L
%pip install -q ultralytics kagglehub

## 2. Imports, seed and device

In [ ]:
# What this cell does: Imports libraries, fixes random seeds and selects the device.
# How it works: Seeds Python/NumPy/PyTorch so the split is reproducible; picks CUDA when available.
import os, random, shutil, hashlib, time, warnings
from pathlib import Path
from collections import Counter

import cv2
import yaml
import numpy as np
import matplotlib.pyplot as plt
import torch
import ultralytics
from IPython.display import Image as IPyImage, display

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("Ultralytics:", ultralytics.__version__)
print("PyTorch:", torch.__version__, "| OpenCV:", cv2.__version__)
print("CUDA available:", torch.cuda.is_available(), "| device:", DEVICE)
if DEVICE == "cpu":
    print("WARNING: no GPU. Training will be very slow. Use Runtime -> Change runtime type -> T4 GPU.")

## 3. Download the dataset from Kaggle

`kagglehub` downloads public datasets without an API key. The first run may take a minute.

In [ ]:
# What this cell does: Downloads the Indian license-plate dataset.
# How it works: kagglehub.dataset_download returns the local folder containing the files.
import kagglehub

# Download latest version
path = kagglehub.dataset_download("kedarsai/indian-license-plates-with-labels")

print("Path to dataset files:", path)

## 4. Explore the folder structure

We do not assume a fixed layout. The next cells **discover** images and labels automatically, so the notebook
also works if the dataset already contains `train/val` folders or uses different folder names.

In [ ]:
# What this cell does: Lists file types and the folders that contain files.
# How it works: Walks the dataset recursively and counts files by extension and by folder.
DATA_ROOT = Path(path)
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

all_files = [p for p in DATA_ROOT.rglob("*") if p.is_file()]
print("Total files:", len(all_files))
print("By extension:", dict(Counter(p.suffix.lower() for p in all_files)))

folders = Counter(p.parent for p in all_files)
print("\nFolders with files:")
for folder, n in sorted(folders.items()):
    print(f"  {n:6d}  {folder.relative_to(DATA_ROOT) if folder != DATA_ROOT else '.'}")

for extra in all_files:
    if extra.suffix.lower() in {".yaml", ".yml", ".csv", ".json"} or extra.name.lower() in {"classes.txt", "readme.md"}:
        print("\nExtra file:", extra.relative_to(DATA_ROOT))
        if extra.suffix.lower() in {".yaml", ".yml"} or extra.name.lower() == "classes.txt":
            print(extra.read_text()[:600])

## 5. Pair every image with its label and validate the annotations

A YOLO label file has one line per object: `class x_center y_center width height`, all normalised to `0–1`.

This cell:
- pairs images and labels (same folder, or `images/` ↔ `labels/` folders),
- reports **images without labels** (the 62 you saw in the first analysis),
- reports **invalid lines** (wrong class, coordinates outside 0–1, zero-size boxes),
- measures how large plates are inside each image.

In [ ]:
# What this cell does: Pairs images with labels and validates every annotation line.
# How it works: Looks for <stem>.txt next to the image or in a parallel "labels" folder, parses each line,
#               and keeps only well-formed boxes of class 0. Everything suspicious is counted and reported.
images = sorted(p for p in all_files if p.suffix.lower() in IMG_EXT)
txt_files = [p for p in all_files if p.suffix.lower() == ".txt" and p.name.lower() not in {"classes.txt", "readme.txt"}]

by_stem = {}
for t in txt_files:
    by_stem.setdefault(t.stem, []).append(t)

def find_label(img: Path):
    cands = [img.with_suffix(".txt")]
    parts = list(img.parts)
    for i, part in enumerate(parts):
        if part.lower() == "images":
            alt = parts.copy(); alt[i] = "labels"
            cands.append(Path(*alt).with_suffix(".txt"))
    for c in cands:
        if c.exists():
            return c
    same = by_stem.get(img.stem, [])
    return same[0] if len(same) == 1 else None      # only trust a global match if it is unique

def parse_label(lbl: Path):
    good, bad = [], 0
    for line in lbl.read_text().strip().splitlines():
        tok = line.split()
        try:
            if len(tok) != 5:
                raise ValueError
            c = int(float(tok[0])); x, y, w, h = map(float, tok[1:])
            if c != 0 or not (0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1):
                raise ValueError
            good.append((0, x, y, w, h))
        except ValueError:
            bad += 1
    return good, bad

labeled, unlabeled, empty_label, bad_lines, class_ids = [], [], [], 0, Counter()
box_w, box_h, per_image = [], [], []
for img in images:
    lbl = find_label(img)
    if lbl is None:
        unlabeled.append(img); continue
    for line in lbl.read_text().strip().splitlines():
        if line.split():
            class_ids[line.split()[0]] += 1
    boxes, bad = parse_label(lbl)
    bad_lines += bad
    if not boxes:
        empty_label.append(img); continue
    labeled.append((img, boxes))
    per_image.append(len(boxes))
    box_w += [b[3] for b in boxes]; box_h += [b[4] for b in boxes]

print(f"Images found            : {len(images)}")
print(f"Images with valid boxes : {len(labeled)}")
print(f"Images with NO label    : {len(unlabeled)}")
print(f"Label files w/o boxes   : {len(empty_label)}")
print(f"Invalid label lines     : {bad_lines}")
print(f"Total valid boxes       : {sum(per_image)}   (max per image: {max(per_image)})")
print(f"Class ids seen in files : {dict(class_ids)}")

box_w, box_h = np.array(box_w), np.array(box_h)
print("\nPlate width as a fraction of image width:")
for q in (5, 25, 50, 75, 95):
    print(f"  p{q:02d}: {np.percentile(box_w, q):.2f}")
close_up = float((box_w > 0.40).mean())
print(f"\nBoxes wider than 40% of the image (close-ups): {close_up:.0%}")
if close_up > 0.5:
    print("-> Mostly close-ups. Real road scenes have plates that are only ~3-15% wide, so we use strong")
    print("   scale augmentation below and run a road-scene stress test after training.")

fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
ax[0].hist(box_w, bins=30, color="#2f9e44"); ax[0].set_title("Plate width / image width"); ax[0].set_xlabel("fraction")
ax[1].hist(box_h, bins=30, color="#1971c2"); ax[1].set_title("Plate height / image height"); ax[1].set_xlabel("fraction")
plt.tight_layout(); plt.show()

## 6. Look at the data (never train on data you have not looked at)

In [ ]:
# What this cell does: Draws 12 random labelled images with their ground-truth boxes.
# How it works: Converts normalised YOLO boxes back to pixels and draws them with OpenCV.
def draw_gt(img_path, boxes):
    im = cv2.imread(str(img_path)); H, W = im.shape[:2]
    for _, x, y, w, h in boxes:
        x1, y1, x2, y2 = int((x - w/2)*W), int((y - h/2)*H), int((x + w/2)*W), int((y + h/2)*H)
        cv2.rectangle(im, (x1, y1), (x2, y2), (0, 200, 0), max(2, W // 200))
    return cv2.cvtColor(im, cv2.COLOR_BGR2RGB)

sample = random.sample(labeled, min(12, len(labeled)))
fig, axes = plt.subplots(3, 4, figsize=(16, 9))
for ax, (img, boxes) in zip(axes.ravel(), sample):
    ax.imshow(draw_gt(img, boxes)); ax.set_title(f"{img.name}  ({len(boxes)} box)", fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# What this cell does: Shows images that have NO label file so you can decide what they are.
# How it works: If they clearly contain a plate, the annotation is missing (keep them OUT of training).
#               If they clearly contain no plate they are valid "negative" images.
if unlabeled:
    show = random.sample(unlabeled, min(8, len(unlabeled)))
    fig, axes = plt.subplots(2, 4, figsize=(16, 6))
    for ax, p in zip(axes.ravel(), show):
        ax.imshow(cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)); ax.set_title(p.name, fontsize=9); ax.axis("off")
    plt.tight_layout(); plt.show()
else:
    print("Every image has a label file.")

## 7. Build a clean YOLO dataset (dedupe + 80 / 10 / 10 split)

Choices made here:
- **Exact duplicate images are removed** (same file hash) so the same picture cannot appear in both train and test.
- Invalid label lines are dropped; all boxes are forced to class `0` (`license_plate`).
- Images **without labels are excluded** by default. Set `INCLUDE_UNLABELED_AS_NEGATIVES = True` only if the
  previous cell showed that they really contain no plate.
- The split is random with a fixed seed.

> Note: the dataset contains several photos of the same plate from different angles. A random split can put
> near-duplicates in both train and test, so test scores may look slightly better than real-world accuracy.

In [ ]:
# What this cell does: Creates /content/anpr_data with images/ and labels/ for train, val and test + data.yaml.
# How it works: Deduplicates by MD5, shuffles with a seed, copies files, writes cleaned labels and a YAML file.
INCLUDE_UNLABELED_AS_NEGATIVES = False
SPLIT = {"train": 0.80, "val": 0.10, "test": 0.10}

def md5(p, chunk=1 << 20):
    h = hashlib.md5()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

seen, unique, dupes, unreadable = set(), [], 0, 0
for img, boxes in labeled:
    if cv2.imread(str(img)) is None:
        unreadable += 1; continue
    h = md5(img)
    if h in seen:
        dupes += 1; continue
    seen.add(h); unique.append((img, boxes))
print(f"Removed duplicates: {dupes} | unreadable images: {unreadable} | kept: {len(unique)}")

random.Random(SEED).shuffle(unique)
n = len(unique); n_test = int(SPLIT["test"] * n); n_val = int(SPLIT["val"] * n)
splits = {"test": unique[:n_test], "val": unique[n_test:n_test + n_val], "train": unique[n_test + n_val:]}

OUT = Path("/content/anpr_data")
if OUT.exists():
    shutil.rmtree(OUT)
for s in splits:
    (OUT / "images" / s).mkdir(parents=True); (OUT / "labels" / s).mkdir(parents=True)

for s, items in splits.items():
    for i, (img, boxes) in enumerate(items):
        name = f"{i:05d}_{img.stem}"
        shutil.copy2(img, OUT / "images" / s / f"{name}{img.suffix.lower()}")
        with open(OUT / "labels" / s / f"{name}.txt", "w") as f:
            for c, x, y, w, h in boxes:
                f.write(f"{c} {x:.6f} {y:.6f} {w:.6f} {h:.6f}\n")

if INCLUDE_UNLABELED_AS_NEGATIVES:
    for i, img in enumerate(unlabeled):
        name = f"neg_{i:04d}"
        shutil.copy2(img, OUT / "images" / "train" / f"{name}{img.suffix.lower()}")
        (OUT / "labels" / "train" / f"{name}.txt").write_text("")
    print("Added", len(unlabeled), "negative images to train")

DATA_YAML = OUT / "data.yaml"
DATA_YAML.write_text(yaml.safe_dump({
    "path": str(OUT), "train": "images/train", "val": "images/val", "test": "images/test",
    "names": {0: "license_plate"},
}, sort_keys=False))

for s in splits:
    print(f"{s:5s}: {len(list((OUT / 'images' / s).glob('*')))} images")
print(DATA_YAML.read_text())

## 8. (Recommended) Save training runs to Google Drive

Colab sessions can disconnect. Saving to Drive keeps `best.pt` and lets you resume. Set `USE_DRIVE = False`
to keep everything in the temporary session storage instead.

In [ ]:
# What this cell does: Optionally mounts Google Drive and chooses where training runs are stored.
# How it works: PROJECT_DIR is passed to YOLO as the "project" folder.
USE_DRIVE = True

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/anpr_runs"
else:
    PROJECT_DIR = "/content/anpr_runs"
os.makedirs(PROJECT_DIR, exist_ok=True)
RUN_NAME = "plate_detector"
print("Runs will be saved in:", PROJECT_DIR)

## 9. Train the detector (transfer learning)

We fine-tune a **pretrained YOLO** model (trained on COCO) — the same idea as using pretrained weights in Lab 11,
except now we also *train* it so it learns one new class: `license_plate`.

Key settings and why:

| Setting | Value | Reason |
|---|---|---|
| model | `yolo11s.pt` | good accuracy/speed balance; use `yolo11n.pt` for a faster app on weak GPUs |
| `imgsz` | 640 | standard size; plates in road scenes are small |
| `scale` | 0.6 | randomly shrinks images — turns close-ups into smaller plates |
| `mosaic` | 1.0 | pastes 4 images together — many small plates on varied backgrounds |
| `fliplr` | 0.0 | mirrored plates are unrealistic, so no horizontal flip |
| `hsv_*` | strong | lighting / colour changes (day, dusk, headlights) |
| `perspective`, `degrees`, `shear` | small | plates seen at an angle |
| `patience` | 15 | stop early when validation stops improving |

Expected time on a T4: roughly 20–40 minutes for 60 epochs.

In [ ]:
# What this cell does: Fine-tunes YOLO on the plate dataset.
# How it works: Loads pretrained weights, trains with augmentations tuned for plates, validates every epoch,
#               and keeps weights/best.pt (highest mAP) and weights/last.pt in PROJECT_DIR/RUN_NAME.
from ultralytics import YOLO

MODEL_NAME = "yolo11s.pt"      # try "yolo11n.pt" (faster) or "yolo11m.pt" (more accurate)
EPOCHS = 60
BATCH = 16                      # lower to 8 if you see an out-of-memory error

model = YOLO(MODEL_NAME)
results = model.train(
    data=str(DATA_YAML), epochs=EPOCHS, imgsz=640, batch=BATCH, device=DEVICE, workers=2,
    project=PROJECT_DIR, name=RUN_NAME, exist_ok=True, seed=SEED, patience=15, cos_lr=True,
    cache="ram", amp=True, plots=True,
    # augmentation (see the table above)
    mosaic=1.0, close_mosaic=10, scale=0.6, translate=0.15, degrees=5, shear=2, perspective=0.0005,
    fliplr=0.0, flipud=0.0, hsv_h=0.015, hsv_s=0.6, hsv_v=0.45, mixup=0.0,
)
BEST = Path(PROJECT_DIR) / RUN_NAME / "weights" / "best.pt"
print("\nBest weights:", BEST, "| exists:", BEST.exists())

## 10. Training curves

What to look for:
- `train/box_loss` and `val/box_loss` should go down together. If val rises while train falls, the model is overfitting.
- `metrics/mAP50(B)` should climb and flatten.

In [ ]:
# What this cell does: Displays the training plots created by YOLO.
# How it works: Opens the PNG files saved in the run folder.
run_dir = Path(PROJECT_DIR) / RUN_NAME
for name in ["results.png", "BoxPR_curve.png", "BoxF1_curve.png", "confusion_matrix_normalized.png"]:
    f = run_dir / name
    if f.exists():
        print(name); display(IPyImage(filename=str(f), width=900))

## 11. Evaluate on the untouched test set and choose a confidence threshold

- **Precision** – of the boxes drawn, how many are real plates.
- **Recall** – of the real plates, how many were found.
- **mAP50** – average precision at IoU 0.50. **mAP50-95** is stricter about box tightness.

The **best-F1 confidence** is a good starting threshold for the real-time app (`detector.conf` in `config.yaml`).
Remember: a confidence score is a model score, not a perfectly calibrated probability.

In [ ]:
# What this cell does: Scores best.pt on the held-out test images and finds the best confidence threshold.
# How it works: model.val(split="test") returns precision/recall/mAP and the F1-vs-confidence curve;
#               the confidence where the smoothed F1 peaks is printed as the recommended threshold.
best = YOLO(str(BEST))
m = best.val(data=str(DATA_YAML), split="test", imgsz=640, conf=0.001, iou=0.6, device=DEVICE,
             plots=True, project=PROJECT_DIR, name="test_eval", exist_ok=True, verbose=False)

print(f"\nTest precision : {m.box.mp:.3f}")
print(f"Test recall    : {m.box.mr:.3f}")
print(f"Test mAP50     : {m.box.map50:.3f}")
print(f"Test mAP50-95  : {m.box.map:.3f}")
print("Speed (ms/img) :", {k: round(v, 1) for k, v in m.speed.items()})

f1 = np.convolve(m.box.f1_curve.mean(0), np.ones(21) / 21, mode="same")
BEST_CONF = float(m.box.px[int(f1.argmax())])
print(f"\nBest-F1 confidence threshold: {BEST_CONF:.2f}  (F1 = {f1.max():.3f})")
print(f"-> put  detector.conf: {max(0.25, round(BEST_CONF, 2))}  in config.yaml")

## 12. Look at predictions with confidence scores

In [ ]:
# What this cell does: Runs the trained detector on 8 random test images and displays boxes + confidence.
# How it works: model.predict(...).plot() draws "license_plate 0.xx" boxes; we show them in a grid.
test_imgs = sorted((OUT / "images" / "test").glob("*"))
pick = random.sample(test_imgs, min(8, len(test_imgs)))
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, p in zip(axes.ravel(), pick):
    r = best.predict(str(p), conf=max(0.25, BEST_CONF), imgsz=640, device=DEVICE, verbose=False)[0]
    ax.imshow(cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB)); ax.axis("off")
    ax.set_title(f"{len(r.boxes)} plate(s)", fontsize=10)
plt.tight_layout(); plt.show()

## 13. Road-scene stress test (important)

The audit showed that most training images are close-ups. A real camera sees small, blurred and dark plates.
This cell simulates that: each test image is **shrunk onto a 1280×720 road-grey canvas** (plate width 40–160 px),
then **motion-blurred, darkened, noised and JPEG-compressed**. We then measure how many plates are still found.

If recall drops sharply for small plates, collect a few real road frames, label them, add them to the dataset
and retrain. This is the most valuable improvement you can make.

In [ ]:
# What this cell does: Measures plate recall on synthetic "road-like" versions of the test images.
# How it works: Rescales so the plate has a chosen pixel width, pastes on a grey canvas, applies blur/dark/noise/JPEG,
#               runs the detector and counts a hit when a predicted box overlaps the ground truth (IoU >= 0.5).
def iou(a, b):
    ix1, iy1, ix2, iy2 = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    ua = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / ua if ua > 0 else 0

def make_road_scene(img, box_norm, rng, W=1280, H=720):
    ih, iw = img.shape[:2]
    _, x, y, w, h = box_norm
    target_w = rng.uniform(40, 160)
    s = target_w / max(w * iw, 1)
    new = cv2.resize(img, (max(8, int(iw * s)), max(8, int(ih * s))), interpolation=cv2.INTER_AREA)
    nh, nw = new.shape[:2]
    canvas = np.full((H, W, 3), rng.integers(70, 130), np.uint8)
    canvas = cv2.add(canvas, rng.normal(0, 8, canvas.shape).astype(np.int16).clip(0, 255).astype(np.uint8))
    ox, oy = int(rng.integers(0, W - nw)), int(rng.integers(0, H - nh))
    canvas[oy:oy + nh, ox:ox + nw] = new
    gt = (ox + (x - w/2) * nw, oy + (y - h/2) * nh, ox + (x + w/2) * nw, oy + (y + h/2) * nh)
    k = int(rng.integers(1, 10))                                    # motion blur
    if k > 1:
        kern = np.zeros((k, k), np.float32); kern[k // 2, :] = 1.0 / k
        canvas = cv2.filter2D(canvas, -1, kern)
    canvas = (255 * (canvas / 255.0) ** rng.uniform(0.9, 2.0)).astype(np.uint8)   # darker
    canvas = cv2.add(canvas, rng.normal(0, 6, canvas.shape).astype(np.int16).clip(0, 255).astype(np.uint8))
    ok, enc = cv2.imencode(".jpg", canvas, [cv2.IMWRITE_JPEG_QUALITY, int(rng.integers(40, 90))])
    return cv2.imdecode(enc, 1), gt, target_w

rng = np.random.default_rng(SEED)
N_STRESS = min(200, len(test_imgs))
bins = {"40-70 px": [], "70-110 px": [], "110-160 px": []}
examples = []
for p in test_imgs[:N_STRESS]:
    lines = (OUT / "labels" / "test" / f"{p.stem}.txt").read_text().strip().splitlines()
    if len(lines) != 1:
        continue
    box = tuple(map(float, lines[0].split()))
    scene, gt, tw = make_road_scene(cv2.imread(str(p)), box, rng)
    r = best.predict(scene, conf=0.25, imgsz=640, device=DEVICE, verbose=False)[0]
    preds = r.boxes.xyxy.cpu().numpy() if len(r.boxes) else np.zeros((0, 4))
    hit = any(iou(gt, pb) >= 0.5 for pb in preds)
    key = "40-70 px" if tw < 70 else "70-110 px" if tw < 110 else "110-160 px"
    bins[key].append(hit)
    if len(examples) < 6:
        examples.append((r.plot(), hit))

print("Recall on road-like scenes (conf 0.25, imgsz 640):")
for k, v in bins.items():
    print(f"  plate width {k:11s}: {np.mean(v):.0%}  ({len(v)} images)" if v else f"  {k}: no samples")

fig, axes = plt.subplots(2, 3, figsize=(18, 7))
for ax, (im, hit) in zip(axes.ravel(), examples):
    ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB)); ax.axis("off"); ax.set_title("found" if hit else "MISSED")
plt.tight_layout(); plt.show()
print("Tip: if the 40-70 px row is low, use imgsz=960 in the app or add real road frames and retrain.")

## 14. Label audit: could the 62 unlabeled images contain plates?

We run the trained detector on the images that had no label file. Confident detections there usually mean the
**annotation is missing** (these images should stay out of training, or be labelled and added).

In [ ]:
# What this cell does: Runs the detector on the unlabeled images and lists the ones where it sees a plate.
# How it works: predict with conf 0.6; images with at least one box are probably missing annotations.
suspect = []
for p in unlabeled:
    r = best.predict(str(p), conf=0.6, imgsz=640, device=DEVICE, verbose=False)[0]
    if len(r.boxes):
        suspect.append((p, r))
print(f"{len(suspect)} of {len(unlabeled)} unlabeled images contain a confident plate detection.")
if suspect:
    fig, axes = plt.subplots(2, 4, figsize=(18, 8))
    for ax, (p, r) in zip(axes.ravel(), suspect[:8]):
        ax.imshow(cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB)); ax.axis("off"); ax.set_title(p.name, fontsize=9)
    plt.tight_layout(); plt.show()

## 15. (Optional) Preview OCR on detected plates

This previews the **second stage** of the real-time system. EasyOCR reads the characters from the cropped plate.
Reading is pretrained, not fine-tuned, so expect some errors; the VS Code project adds Indian-format repair and
voting over frames to fix many of them.

In [ ]:
# What this cell does: Crops detected plates from test images and reads them with EasyOCR.
# How it works: best.predict -> crop box -> upscale + CLAHE -> easyocr.readtext -> show crop with the text.
%pip install -q easyocr
import easyocr
reader = easyocr.Reader(["en"], gpu=torch.cuda.is_available(), verbose=False)

fig, axes = plt.subplots(2, 4, figsize=(16, 6))
for ax, p in zip(axes.ravel(), random.sample(test_imgs, min(8, len(test_imgs)))):
    im = cv2.imread(str(p))
    r = best.predict(im, conf=0.4, imgsz=640, device=DEVICE, verbose=False)[0]
    if not len(r.boxes):
        ax.axis("off"); continue
    x1, y1, x2, y2 = map(int, r.boxes.xyxy[0].tolist()); conf = float(r.boxes.conf[0])
    crop = im[max(0, y1):y2, max(0, x1):x2]
    g = cv2.cvtColor(cv2.resize(crop, None, fx=max(1, 64 / crop.shape[0]), fy=max(1, 64 / crop.shape[0])), cv2.COLOR_BGR2GRAY)
    g = cv2.createCLAHE(2.0, (4, 4)).apply(g)
    out = reader.readtext(g, allowlist="ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789")
    text = " ".join(t for _, t, _ in out); oc = np.mean([c for *_, c in out]) if out else 0
    ax.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)); ax.axis("off")
    ax.set_title(f"{text}\ndet {conf:.0%} | ocr {oc:.0%}", fontsize=11)
plt.tight_layout(); plt.show()

## 16. Export and download `best.pt`

Copy the file into your VS Code project at `anpr-realtime/weights/best.pt`.

An optional ONNX export is also created (useful for CPU-only machines with `onnxruntime`).

In [ ]:
# What this cell does: Copies best.pt to /content, optionally exports ONNX, and downloads the weights.
# How it works: files.download() sends the file to your browser; the Drive copy already exists if USE_DRIVE=True.
shutil.copy2(BEST, "/content/best.pt")
print(f"best.pt size: {Path('/content/best.pt').stat().st_size / 1e6:.1f} MB")

try:
    onnx_path = best.export(format="onnx", imgsz=640, simplify=True)
    print("ONNX exported:", onnx_path)
except Exception as e:
    print("ONNX export skipped:", e)

print("\nRecommended settings for config.yaml:")
print(f"  detector.conf : {max(0.25, round(BEST_CONF, 2))}")
print( "  detector.imgsz: 640  (960 if small/far plates are missed in the stress test)")

from google.colab import files
files.download("/content/best.pt")

## 17. Summary and next steps

1. Copy `best.pt` into `anpr-realtime/weights/` in VS Code.
2. Run `python main.py --source 0` for the live webcam or `--source video.mp4` for a video.
3. If the stress test (section 13) showed weak recall on small plates: add 100–300 real road frames, label them
   (Roboflow or CVAT, class `license_plate`), merge them into the dataset and retrain.
4. For better text accuracy, fine-tune an OCR model on plate crops saved by the app (`outputs/plate_crops/`).

**Limitations to remember:** detection scores are not calibrated probabilities; night, rain, motion blur and
dirty or non-standard plates reduce accuracy; and number-plate recognition processes personal data, so use it
only where you are permitted to do so.